In [58]:
from pathlib import Path
import csv

In [59]:
def write_to_file(file_name, output):
    with open(file_name, "a", encoding="utf-8") as text_file:
        text_file.write("\n".join(output)+ "\n")

In [60]:
def print_stackX2(telescope, date, img_range_start, img_range_end, fwhm_lim):
    return (f"python /mnt/64bin/stackX2.py {telescope} {date} {img_range_start} {img_range_end} {fwhm_lim}")

In [69]:
def print_rename_copy(supernova_name, obv_date, telescope, band):
    new_file_name = f"{supernova_name}-{obv_date}-{telescope}-{band}-mosaic.fits"
    new_dir = "../../../../output_mosaics_flats"

    return (
        f"if ( -f amosaic.fits ) then\n"
        f"    mv amosaic.fits {new_file_name}\n"
        f"    cp {new_file_name} {new_dir}/\n"
        "else\n"
        '    echo "No amosaic.fits; skipping this group"\n'
        "endif"
       )
    
    
    #if amosaic.fits exists:
        #return("rename amosaic.fits new_file_name")
        #return("copy new_file_name new_dir")if 
    
    
    #print(new_file_name)


    
#print_rename_copy("sn2026aaiv", "26_10_13", "pt5m", "B")

In [62]:
def print_mosaic_folder(telescope, band):
    folder_name = f"{telescope}_{band}"
    return(
        f"mkdir {folder_name}\n"
        f"cd {folder_name}\n"
    )


In [63]:
def print_day_folder(obv_date):
    return(
        f"mkdir {obv_date}\n"
        f"cd {obv_date}\n"
    )

In [64]:
def read_obv_rows(file_path):
    with open(file_path, encoding="utf-8") as file:
        groups = []
        
        next(file) 

        for line in file:
            if line.strip():
                groups.append(line.split())

    return groups

In [65]:
def find_flats(telescope, band, obv_date):
    band = band.upper()
    if telescope == "pt5m":
        if obv_date in ["26_09_24", "26_10_02", "26_10_03",
                        "26_10_04", "26_10_06", "26_10_07",
                        "26_10_13"]:

            return(
                f'cp "/mnt/archive/pt5m/2026/{obv_date}/calibs/{band}flatnorm2x2.fits" master_flats.fits'
            )
        
        else:
            return(
                f'cp "/mnt/archive/pt5m/2026/26_10_03/calibs/{band}flatnorm2x2.fits" master_flats.fits'
            )
    
    elif telescope == "east_16":
        return(
            f'cp "/mnt/share/flatfields/flats_Eph2026/east-16_26_02_24_{band}_master_flat.fits" master_flats.fits'
        )
    elif telescope == "draco2":
        return(
            f'cp "/mnt/share/flatfields/flats_Eph2026/draco2_26_02_24_{band}_master_flat.fits" master_flats.fits'
        )
    elif telescope == "west_14":
        return(
            f'cp "/mnt/share/flatfields/flats_Eph2025/west14/{band.lower()}/master_flat_west-14_25_01_25_{band}.fits" master_flats.fits'
        )

        
                
                
            
    
    

In [66]:
def main_func(master_list, output_file):
    #master_list = f"{supernova}_list"
    #output_file = f"{supernova}_output"
    for files in master_list:
        output = []

        #supernova_name = #name from file
        #obv_date = #date from filename

        filename = Path(files)
        supernova_name, obv_date = filename.stem.rsplit("-", 1)

        output.append(print_day_folder(obv_date))

        rows = read_obv_rows(files)

        for obv in rows:
            #telescope = #telescope from that row
            #band = #band from that row
            #img_range_start = #from row
            #img_range_end = #from row

            telescope, band, exposure, count, img_range_start, img_range_end, fwhm = obv

            
            output.append(print_mosaic_folder(telescope, band))

            # Here is where we put the func to copy over and rename the pt5m flatfield
            output.append(find_flats(telescope, band, obv_date))
            
            output.append(print_stackX2(telescope, obv_date, img_range_start, img_range_end, "5.0"))
            output.append(print_rename_copy(supernova_name, obv_date,telescope, band))

            output.append("cd ..")

        output.append("cd ..")
        
        write_to_file(f"{output_file}.txt", output)
        

In [70]:
sn2026aaiv_list = [
    'Batch Creation/obv data/sn2026aaiv/sn2026aaiv-26_09_24.txt',
    'Batch Creation/obv data/sn2026aaiv/sn2026aaiv-26_09_28.txt',
    'Batch Creation/obv data/sn2026aaiv/sn2026aaiv-26_10_01.txt',
    'Batch Creation/obv data/sn2026aaiv/sn2026aaiv-26_10_02.txt',
    'Batch Creation/obv data/sn2026aaiv/sn2026aaiv-26_10_03.txt',
    'Batch Creation/obv data/sn2026aaiv/sn2026aaiv-26_10_04.txt',
    'Batch Creation/obv data/sn2026aaiv/sn2026aaiv-26_10_05.txt',
    'Batch Creation/obv data/sn2026aaiv/sn2026aaiv-26_10_06.txt',
    'Batch Creation/obv data/sn2026aaiv/sn2026aaiv-26_10_07.txt',
    'Batch Creation/obv data/sn2026aaiv/sn2026aaiv-26_10_09.txt',
    'Batch Creation/obv data/sn2026aaiv/sn2026aaiv-26_10_10.txt',
    'Batch Creation/obv data/sn2026aaiv/sn2026aaiv-26_10_11.txt',
    'Batch Creation/obv data/sn2026aaiv/sn2026aaiv-26_10_12.txt',
    'Batch Creation/obv data/sn2026aaiv/sn2026aaiv-26_10_13.txt',
    'Batch Creation/obv data/sn2026aaiv/sn2026aaiv-26_10_16.txt',
]

sn2026acna_list = [
    'Batch Creation/obv data/sn2026acna/sn2026acna-26_09_24.txt',
    'Batch Creation/obv data/sn2026acna/sn2026acna-26_09_29.txt',
    'Batch Creation/obv data/sn2026acna/sn2026acna-26_10_01.txt',
    'Batch Creation/obv data/sn2026acna/sn2026acna-26_10_02.txt',
    'Batch Creation/obv data/sn2026acna/sn2026acna-26_10_03.txt',
    'Batch Creation/obv data/sn2026acna/sn2026acna-26_10_05.txt',
    'Batch Creation/obv data/sn2026acna/sn2026acna-26_10_06.txt',
    'Batch Creation/obv data/sn2026acna/sn2026acna-26_10_07.txt',
    'Batch Creation/obv data/sn2026acna/sn2026acna-26_10_09.txt',
    'Batch Creation/obv data/sn2026acna/sn2026acna-26_10_10.txt',
    'Batch Creation/obv data/sn2026acna/sn2026acna-26_10_11.txt',
    'Batch Creation/obv data/sn2026acna/sn2026acna-26_10_12.txt',
    'Batch Creation/obv data/sn2026acna/sn2026acna-26_10_13.txt',
    'Batch Creation/obv data/sn2026acna/sn2026acna-26_10_16.txt',
    'Batch Creation/obv data/sn2026acna/sn2026acna-26_10_17.txt',
]

sn2026aeiv_list = [
    'Batch Creation/obv data/sn2026aeiv/sn2026aeiv-26_10_09.txt',
    'Batch Creation/obv data/sn2026aeiv/sn2026aeiv-26_10_10.txt',
    'Batch Creation/obv data/sn2026aeiv/sn2026aeiv-26_10_11.txt',
    'Batch Creation/obv data/sn2026aeiv/sn2026aeiv-26_10_12.txt',
    'Batch Creation/obv data/sn2026aeiv/sn2026aeiv-26_10_13.txt',
    'Batch Creation/obv data/sn2026aeiv/sn2026aeiv-26_10_16.txt',
]

at2026aeiz_list = [
    'Batch Creation/obv data/at2026aeiz/at2026aeiz-26_10_13.txt',
] 

at2026aesj_list = [
    'Batch Creation/obv data/at2026aesj/at2026aesj-26_10_13.txt',
]

sn2026acpp_list = [
    'Batch Creation/obv data/sn2026acpp/sn2026acpp-26_10_03.txt',
    'Batch Creation/obv data/sn2026acpp/sn2026acpp-26_10_05.txt',
    'Batch Creation/obv data/sn2026acpp/sn2026acpp-26_10_06.txt',
    'Batch Creation/obv data/sn2026acpp/sn2026acpp-26_10_10.txt',
    'Batch Creation/obv data/sn2026acpp/sn2026acpp-26_10_11.txt',
]

In [71]:
main_func(sn2026aaiv_list, "sn2026aaiv_output")
main_func(sn2026acna_list, "sn2026acna_output")
main_func(sn2026aeiv_list, "sn2026aeiv_output")
main_func(at2026aeiz_list, "at2026aeiz_output")
main_func(at2026aesj_list, "at2026aesj_output")
main_func(sn2026acpp_list, "sn2026acpp_output")